# Service Access Cleaning
Prepare exactly three indicators for 2021–2025:
- Water: At least basic drinking-water access — reported 105 rows.
- Electricity: Electricity access — reported 15 rows.
- Internet: Internet access at home — reported 25 rows.

Expected baseline: 145 rows, including missing observations. Exclude Internet use and Internet use, age 10+ along with all other indicators.
Supports BP/AP3 as service-access context; AP1 requires spending data and AP2 requires compatible execution data.
Temporary views only. No Silver table is written. Run cells in order on the same compute session.
Source columns follow the supplied Bronze screenshots. SQL has not been executed against Databricks.


## 1. Clean text and assign services
Trim text and normalize blanks to NULL. Preserve source fields temporarily for validation and stable ID generation.
`service_measure` retains the specific indicator name. Selected services exclude sanitation, handwashing and clean cooking.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW service_access_clean_text AS
WITH cleaned AS (
 SELECT
  NULLIF(TRIM(country), '') AS country,
  NULLIF(TRIM(year), '') AS year_raw,
  TRY_CAST(year AS INT) AS year,
  NULLIF(TRIM(region), '') AS region_raw,
  NULLIF(TRIM(indicator), '') AS service_measure,
  NULLIF(TRIM(breakdown_type), '') AS breakdown_type,
  NULLIF(TRIM(breakdown_value), '') AS breakdown_value,
  NULLIF(TRIM(value), '') AS value_raw,
  NULLIF(TRIM(unit), '') AS unit,
  NULLIF(TRIM(source), '') AS source,
  NULLIF(TRIM(notes), '') AS notes,
  _rescued_data, _source_file, _run_id, _ingested_at
 FROM pondolens.dev_bronze.service_access
)
SELECT *, CASE
 WHEN LOWER(service_measure) LIKE 'at least basic drinking-water%' THEN 'Water'
 WHEN service_measure = 'Electricity access' THEN 'Electricity'
 WHEN service_measure = 'Internet access at home' THEN 'Internet'
END AS service
FROM cleaned;

## 2. Review scope and years
Expected baseline: 145 selected rows based on earlier profiling, including missing observations. Review the full drinking-water label because screenshots truncated it.
Investigate missing/invalid years before excluding them. NULL service marks an indicator outside the selected scope.


In [0]:
%sql
SELECT service, service_measure, year, COUNT(*) AS row_count
FROM service_access_clean_text
GROUP BY service, service_measure, year
ORDER BY service, service_measure, year;

In [0]:
%sql
SELECT service, service_measure, COUNT(*) AS row_count
FROM service_access_clean_text
WHERE service IS NOT NULL
GROUP BY service, service_measure
ORDER BY service;


In [0]:
%sql
SELECT * FROM service_access_clean_text
WHERE service IS NOT NULL AND (year IS NULL OR year NOT BETWEEN 2021 AND 2025);

## 3. Select scope and cast values
Select 2021–2025 and cast values to DECIMAL. Do not replace missing values with zero or silently strip nonnumeric tokens.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW service_access_clean_values AS
SELECT *, TRY_CAST(value_raw AS DECIMAL(18,6)) AS value
FROM service_access_clean_text
WHERE service IS NOT NULL AND year BETWEEN 2021 AND 2025;

## 4. Validate records
Required fields: country, year, geography label, indicator, unit and source. Breakdown fields may be NULL.
Missing observations are allowed. Nonblank parse failures, percentages outside 0–100 and rescued data require review.
The status shows the first applicable issue; detailed rows remain available for inspection.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW service_access_quality AS
SELECT *, CASE
 WHEN NULLIF(TRIM(_rescued_data), '') IS NOT NULL THEN 'RESCUED_DATA'
 WHEN country IS NULL OR region_raw IS NULL OR service_measure IS NULL
   OR unit IS NULL OR source IS NULL THEN 'MISSING_REQUIRED_FIELD'
 WHEN value_raw IS NOT NULL AND value IS NULL THEN 'INVALID_NUMERIC_VALUE'
 WHEN INSTR(unit, '%') > 0 AND (value < 0 OR value > 100) THEN 'OUT_OF_RANGE'
 WHEN value_raw IS NULL THEN 'MISSING_OBSERVATION'
 ELSE 'VALID'
END AS quality_status
FROM service_access_clean_values;

In [0]:
%sql
SELECT quality_status, COUNT(*) AS row_count
FROM service_access_quality GROUP BY quality_status ORDER BY quality_status;

In [0]:
%sql
SELECT * FROM service_access_quality
WHERE quality_status NOT IN ('VALID','MISSING_OBSERVATION');

## 5. Check duplicate observations
Review the proposed business key. Do not automatically remove duplicates; inspect source revisions and conflicting values.
Repeat this check after geography mapping because different raw labels can map to the same region code.


In [0]:
%sql
SELECT country, year, region_raw, service_measure,
 breakdown_type, breakdown_value, unit, source,
 COUNT(*) AS row_count,
 COUNT(DISTINCT value_raw) AS distinct_nonnull_values,
 SUM(CASE WHEN value_raw IS NULL THEN 1 ELSE 0 END) AS missing_value_rows
FROM service_access_clean_values
GROUP BY country, year, region_raw, service_measure,
 breakdown_type, breakdown_value, unit, source
HAVING COUNT(*) > 1;

## 6. Prepare geography and definition mapping
Use the same effective-dated crosswalk as spending, population and unemployment. National observations stay distinct from regional observations.
Do not infer codes or retrospectively change boundaries. Families, households and population are different denominators.


In [0]:
%sql
SELECT DISTINCT country, year, region_raw
FROM service_access_clean_values ORDER BY country, year, region_raw;

In [0]:
%sql
SELECT DISTINCT service, service_measure, unit,
 breakdown_type, breakdown_value, source
FROM service_access_clean_values ORDER BY service, service_measure, unit;

## 7. Summarize completeness
Count observations separately from missing values and parse failures. Do not sum percentages or fill 2025 using 2024.
Parsed values can still be outside the permitted range; consult the quality-status output.


In [0]:
%sql
SELECT service, year, COUNT(*) AS row_count, COUNT(value) AS parsed_values,
 SUM(CASE WHEN value_raw IS NULL THEN 1 ELSE 0 END) AS missing_values,
 SUM(CASE WHEN value_raw IS NOT NULL AND value IS NULL THEN 1 ELSE 0 END) AS invalid_numeric_values
FROM service_access_clean_values GROUP BY service, year ORDER BY service, year;

In [0]:
%sql
SELECT 'selected_before_year_filter' AS stage, COUNT(*) AS row_count
FROM service_access_clean_text WHERE service IS NOT NULL
UNION ALL
SELECT 'selected_2021_2025', COUNT(*) FROM service_access_clean_values;

## 8. Cleaning preview and next steps
The preview is not a final Silver table. Keep numeric `value` for BP/AP3 analysis even though it is absent from the latest draft diagram.
Next steps:
- Configure the actual shared region crosswalk; require exactly one geography match per observation and no row multiplication.
- Generate stable SHA-256 IDs from standardized country/year/geography/indicator/breakdown/unit/source fields. Exclude value and processing metadata.
- Validate mapped business keys, numeric values, required fields and source-to-Silver preservation.
- Preserve source/unit distinctions through a lineage or indicator mapping if omitted from the final Silver schema.
- Generate `processed_at` as the UTC Silver processing timestamp at write time; do not rename the Bronze `_ingested_at` value.
- Aggregate spending separately and select compatible indicator/source/unit/breakdown before analytical joins.
No write cell is included because geography mapping and final numeric-value schema remain unresolved.


In [0]:
%sql
SELECT year, service, service_measure, region_raw,
 breakdown_type, breakdown_value, value, unit, source, quality_status
FROM service_access_quality
ORDER BY service, year, region_raw, service_measure LIMIT 100;